# 20 - The multi-league pipeline: does the pooled model ship?

Notebook 18 found that one logistic regression pooled over the top 5 leagues
beats a model per league. That was on research data (Understat xG, names
matched on the fly). This notebook re-runs the comparison on the
**pipeline's own data and code**, the gate for adding La Liga, the Bundesliga,
Serie A and Ligue 1 to `ENABLED` (docs/multi-league-plan.md, Phase 1).

- **Data** - `load_data.build_dataset` over E0, SP1, D1, I1, F1 from the
  cached raw files (`refresh_current=False`, so no network): football-data
  results, FotMob xG with football-data / Understat as backup, names through
  `team_names.csv`.
- **Features** - `EloDiff`, `XGDDiffLast5`, `GoalsPerGameDiff`, built per
  competition (`scope="competition"`, P1-04) by the pipeline's builders.
- **Models** - the deployed calibrated LR (`model.make_model`):
  - **EPL-only** - today's model, trained on the Premier League alone;
  - **per league** - the same, one model per league;
  - **pooled** - one model on all five, plus `League_<Div>` indicators
    (E0 baseline), as `train_model` now does.
- **Evaluation** - `evaluate.py`: walk-forward by season, test 22-23 to 25-26,
  per-match log loss, paired bootstrap (5,000 resamples, seed 42).

**Gates:**

1. EPL, 1,520 test matches: pooled vs EPL-only must be **non-inferior**
   (95% CI upper bound < +0.002). If this fails, the pooled model doesn't ship.
2. All top-5 test matches (~7,080): pooled vs per-league must be **better**
   (CI upper bound < 0), replicating nb 18's -0.0028 [-0.0044, -0.0012].

In [1]:
import os
import sys
import tempfile
import warnings
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
os.chdir(ROOT)  # the pipeline reads data/ relative to the repo root
sys.path.insert(0, str(ROOT / "src"))

import numpy as np
import pandas as pd
from sklearn.metrics import accuracy_score, log_loss

import competitions
from load_data import build_dataset
from features import build_goals_per_game_feature, build_xgd_last5_feature
from elo import build_elo_feature
from model import FEATURES, LEAGUE_COLUMNS, league_columns, make_model
from backtest import bet_profit
from evaluate import (
    walk_forward, per_match_log_loss, paired_bootstrap, seasons_better,
    better, non_inferior, NON_INFERIOR_MARGIN,
)

warnings.filterwarnings("ignore")
pd.set_option("display.precision", 4)
pd.set_option("display.width", 200)

DIVS = ["E0", "SP1", "D1", "I1", "F1"]
TEST_SEASONS = ["22-23", "23-24", "24-25", "25-26"]
LABELS = ["A", "D", "H"]  # model.classes_ order
NAMES = {div: competitions.get(div)["name"] for div in DIVS}

## Build the dataset with the pipeline

In [2]:
with tempfile.TemporaryDirectory() as tmp:
    matches = build_dataset(DIVS, output_path=Path(tmp) / "all_matches.csv", refresh_current=False)

matches = matches.sort_values(["MatchDateTime", "HomeTeam", "AwayTeam"]).reset_index(drop=True)

# 26-27 is in progress; it stays in for feature history, never as a test season
coverage = (
    matches.assign(HasXG=matches["HomeXG"].notna() & matches["AwayXG"].notna())
    .groupby(["Div", "Season"])
    .agg(Games=("FTR", "size"), XG=("HasXG", "mean"))
    .unstack("Season")
)
coverage

Built 9359 matches (100.0% xG coverage, 1 from football-data/Understat backup)


Saved C:\Users\harry\AppData\Local\Temp\tmplwicex1g\all_matches.csv


Games                                  XG                              
Season 21-22 22-23 23-24 24-25 25-26 26-27 21-22 22-23 23-24 24-25 25-26 26-27
Div                                                                           
D1       306   306   306   306   306    72   1.0   1.0   1.0   1.0   1.0   1.0
E0       380   380   380   380   380    89   1.0   1.0   1.0   1.0   1.0   1.0
F1       380   380   306   306   306    81   1.0   1.0   1.0   1.0   1.0   1.0
I1       380   380   380   380   380   100   1.0   1.0   1.0   1.0   1.0   1.0
SP1      380   380   380   380   380   109   1.0   1.0   1.0   1.0   1.0   1.0

In [3]:
matches.groupby(["Div", "XGSource"]).size().unstack(fill_value=0)

XGSource,FotMob,Understat
Div,,
D1,1602,0
E0,1989,0
F1,1759,0
I1,2000,0
SP1,2008,1


## Features

Built once over the pooled frame. P1-04 showed each league's features are
bit-identical to building it alone, so the EPL-only reference uses the same
E0 rows.

In [4]:
build_elo_feature(matches)
build_goals_per_game_feature(matches)
build_xgd_last5_feature(matches)

matches = league_columns(matches)
INDICATORS = [c for c in LEAGUE_COLUMNS.values() if c in matches.columns]

test = matches[matches["Season"].isin(TEST_SEASONS)]
e0_test = test[test["Div"] == "E0"]
print(f"{len(test)} test matches, {len(e0_test)} in the Premier League; indicators: {INDICATORS}")

7082 test matches, 1520 in the Premier League; indicators: ['League_SP1', 'League_D1', 'League_I1', 'League_F1']


## Walk-forward predictions

In [5]:
e0 = matches[matches["Div"] == "E0"]

probas = {
    "EPL-only": walk_forward(e0, make_model, FEATURES, TEST_SEASONS),
    "Per league": walk_forward(matches, make_model, FEATURES, TEST_SEASONS, group_by="Div"),
    "Pooled": walk_forward(matches, make_model, FEATURES + INDICATORS, TEST_SEASONS),
    "Pooled, no indicators": walk_forward(matches, make_model, FEATURES, TEST_SEASONS),
}

# Per league on E0 is the EPL-only model, trained on exactly the same rows
assert np.allclose(probas["Per league"].loc[e0_test.index], probas["EPL-only"].loc[e0_test.index])

In [6]:
def demargin(row, prefix):
    odds = np.array([row[f"{prefix}{o}"] for o in LABELS], dtype=float)
    implied = 1 / odds
    return implied / implied.sum()


def bookmaker(frame):
    """De-margined Pinnacle closing odds, falling back to the market-average close."""
    rows = []
    for _, row in frame.iterrows():
        prefix = "PSC" if row[["PSCA", "PSCD", "PSCH"]].notna().all() else "AvgC"
        rows.append(demargin(row, prefix))
    return pd.DataFrame(rows, index=frame.index, columns=LABELS)


probas["Bookmaker"] = bookmaker(test)

# Each league's home/draw/away rates from the seasons before the test season
base = []
for season in TEST_SEASONS:
    prior = matches[matches["Season"] < season]
    rates = prior.groupby("Div")["FTR"].value_counts(normalize=True).unstack()[LABELS]
    held = test[test["Season"] == season]
    base.append(pd.DataFrame(rates.loc[held["Div"]].to_numpy(), index=held.index, columns=LABELS))
probas["Base rates"] = pd.concat(base).loc[test.index]

## Scores by league

In [7]:
def scores(frame, proba):
    proba = proba.loc[frame.index]
    pick = proba.idxmax(axis=1)
    profit = sum(bet_profit(m, p) for (_, m), p in zip(frame.iterrows(), pick))
    return {
        "matches": len(frame),
        "logLoss": log_loss(frame["FTR"], proba[LABELS], labels=LABELS),
        "accuracy": accuracy_score(frame["FTR"], pick) * 100,
        "roi": profit / len(frame) * 100,
    }


models = ["Base rates", "Per league", "Pooled, no indicators", "Pooled", "Bookmaker"]
rows = []
for div in DIVS + ["All"]:
    frame = test if div == "All" else test[test["Div"] == div]
    for name in models:
        rows.append({"League": NAMES.get(div, "All five"), "Model": name, **scores(frame, probas[name])})

by_league = pd.DataFrame(rows)
by_league.pivot(index="League", columns="Model", values="logLoss")[models].loc[[*NAMES.values(), "All five"]]

Model,Base rates,Per league,"Pooled, no indicators",Pooled,Bookmaker
League,,,,,
Premier League,1.0689,0.9799,0.9784,0.9798,0.9605
La Liga,1.0639,0.9911,0.9872,0.9847,0.9595
Bundesliga,1.0749,0.9991,0.9931,0.9935,0.9697
Serie A,1.0869,0.9948,0.9928,0.9928,0.9670
Ligue 1,1.0706,0.9989,0.9982,0.9982,0.9784
All five,1.0731,0.9923,0.9896,0.9894,0.9666


In [8]:
(
    by_league[by_league["Model"].isin(["Per league", "Pooled"])]
    .pivot(index="League", columns="Model", values=["accuracy", "roi"])
    .loc[[*NAMES.values(), "All five"]]
    .round(1)
)

accuracy               roi       
Model          Per league Pooled Per league Pooled
League                                            
Premier League       54.7   54.5        0.4    0.4
La Liga              52.3   53.6       -1.4    1.6
Bundesliga           51.6   51.6       -6.4   -6.6
Serie A              52.1   52.5       -3.5   -2.6
Ligue 1              51.6   51.8       -4.8   -4.1
All five             52.5   52.9       -2.9   -2.0

## Gate 1 - EPL: pooled vs EPL-only (non-inferior)

Per-match log-loss difference, pooled - EPL-only, on the 1,520 Premier League
test matches. Negative = pooled better.

In [9]:
def compare(frame, candidate, reference):
    y = frame["FTR"].to_numpy()
    diff = (
        per_match_log_loss(y, probas[candidate].loc[frame.index][LABELS].to_numpy(), LABELS)
        - per_match_log_loss(y, probas[reference].loc[frame.index][LABELS].to_numpy(), LABELS)
    )
    result = paired_bootstrap(diff)
    return {
        "matches": len(frame),
        "mean": result["mean"],
        "ciLow": result["ci"][0],
        "ciHigh": result["ci"][1],
        "pBetter": result["p_better"],
        "seasonsBetter": seasons_better(
            y, probas[candidate].loc[frame.index][LABELS].to_numpy(),
            probas[reference].loc[frame.index][LABELS].to_numpy(),
            frame["Season"].to_numpy(), labels=LABELS,
        ),
        "better": better(result["ci"]),
        "nonInferior": non_inferior(result["ci"]),
    }


gate1 = compare(e0_test, "Pooled", "EPL-only")
print(f"Pooled - EPL-only on {gate1['matches']} EPL matches: {gate1['mean']:+.4f} "
      f"[{gate1['ciLow']:+.4f}, {gate1['ciHigh']:+.4f}], P(better) {gate1['pBetter']:.2f}, "
      f"better in {gate1['seasonsBetter']}/{len(TEST_SEASONS)} seasons")
print(f"Gate 1 (non-inferior, upper < +{NON_INFERIOR_MARGIN}): {'PASS' if gate1['nonInferior'] else 'FAIL'}")

Pooled - EPL-only on 1520 EPL matches: -0.0001 [-0.0028, +0.0025], P(better) 0.53, better in 2/4 seasons
Gate 1 (non-inferior, upper < +0.002): FAIL


## Gate 2 - top 5: pooled vs per league (better)

In [10]:
gate2 = compare(test, "Pooled", "Per league")
print(f"Pooled - per league on {gate2['matches']} matches: {gate2['mean']:+.4f} "
      f"[{gate2['ciLow']:+.4f}, {gate2['ciHigh']:+.4f}], P(better) {gate2['pBetter']:.2f}, "
      f"better in {gate2['seasonsBetter']}/{len(TEST_SEASONS)} seasons")
print(f"Gate 2 (better, upper < 0): {'PASS' if gate2['better'] else 'FAIL'}")
print("nb 18 (Understat data): -0.0028 [-0.0044, -0.0012]")

Pooled - per league on 7082 matches: -0.0029 [-0.0046, -0.0013], P(better) 1.00, better in 4/4 seasons
Gate 2 (better, upper < 0): PASS
nb 18 (Understat data): -0.0028 [-0.0044, -0.0012]


By league, and the indicators on their own (not gates - for the record):

In [11]:
detail = []
for div in DIVS:
    frame = test[test["Div"] == div]
    detail.append({"Comparison": "Pooled - per league", "League": NAMES[div], **compare(frame, "Pooled", "Per league")})
detail.append({"Comparison": "Pooled - pooled without indicators", "League": "All five",
               **compare(test, "Pooled", "Pooled, no indicators")})
pd.DataFrame(detail).drop(columns=["better", "nonInferior"])

,Comparison,League,matches,mean,ciLow,ciHigh,pBetter,seasonsBetter
0,Pooled - per league,Premier League,1520,-9.1232e-05,-0.0028,0.0025,0.5312,2
1,Pooled - per league,La Liga,1520,-6.4013e-03,-0.0108,-0.0020,0.9980,3
2,Pooled - per league,Bundesliga,1224,-5.6265e-03,-0.0105,-0.0008,0.9896,4
3,Pooled - per league,Serie A,1520,-2.0048e-03,-0.0049,0.0009,0.9090,3
4,Pooled - per league,Ligue 1,1298,-6.9423e-04,-0.0034,0.0022,0.6794,2
5,Pooled - pooled without indicators,All five,7082,-1.9690e-04,-0.0014,0.0010,0.6304,3


## Season by season

In [12]:
pd.DataFrame({
    name: {
        season: log_loss(test.loc[test["Season"] == season, "FTR"],
                         probas[name].loc[test.index[test["Season"] == season]][LABELS], labels=LABELS)
        for season in TEST_SEASONS
    }
    for name in ["Per league", "Pooled", "Bookmaker"]
}).assign(**{"Pooled - per league": lambda t: t["Pooled"] - t["Per league"]})

,Per league,Pooled,Bookmaker,Pooled - per league
22-23,0.9972,0.9935,0.9749,-0.0037
23-24,0.9804,0.9762,0.9525,-0.0042
24-25,0.9871,0.9852,0.9606,-0.0019
25-26,1.0042,1.0024,0.9780,-0.0018


## League indicator coefficients

The pooled model trained on 21-22 to 25-26, coefficients averaged over the
5 calibration folds (log-odds, per outcome, relative to the Premier League).

In [13]:
train = matches[matches["Season"] < "26-27"]
fitted = make_model().fit(train[FEATURES + INDICATORS], train["FTR"])
coef = np.mean([c.estimator.coef_ for c in fitted.calibrated_classifiers_], axis=0)
pd.DataFrame(coef, index=fitted.classes_, columns=FEATURES + INDICATORS).T.round(3)

,A,D,H
EloDiff,-0.003,0.000,0.002
XGDDiffLast5,-0.099,-0.070,0.170
GoalsPerGameDiff,-0.103,0.046,0.057
League_SP1,-0.129,0.076,0.053
League_D1,-0.028,0.028,0.000
League_I1,0.013,0.108,-0.121
League_F1,0.018,0.008,-0.026


## After the gate (not gates - chosen after seeing gate 1)

Gate 1 failed on its upper bound, with a mean of about zero. Two questions for
the owner's decision, answered here and **not** used to pass anything:

1. Could a model exactly as good as today's pass this gate on 1,520 matches?
2. What would the alternatives look like?

In [14]:
y = e0_test["FTR"].to_numpy()
diff = (
    per_match_log_loss(y, probas["Pooled"].loc[e0_test.index][LABELS].to_numpy(), LABELS)
    - per_match_log_loss(y, probas["EPL-only"].loc[e0_test.index][LABELS].to_numpy(), LABELS)
)
sd = diff.std(ddof=1)
se = sd / np.sqrt(len(diff))
print(f"Per-match difference SD {sd:.4f}; SE of the mean {se:.5f}; CI half-width ~{1.96 * se:.4f}")
print(f"To pass at n = {len(diff)}, the pooled model must be better by > {1.96 * se - NON_INFERIOR_MARGIN:.4f}")
print(f"A model exactly as good as today's needs n ~ {(1.96 * sd / NON_INFERIOR_MARGIN) ** 2:,.0f} EPL matches to pass")

Per-match difference SD 0.0515; SE of the mean 0.00132; CI half-width ~0.0026
To pass at n = 1520, the pooled model must be better by > 0.0006
A model exactly as good as today's needs n ~ 2,549 EPL matches to pass


In [15]:
# Hybrid: today's EPL-only model for the Premier League, the pooled model for the other four
probas["Hybrid"] = probas["Pooled"].copy()
probas["Hybrid"].loc[e0_test.index] = probas["EPL-only"].loc[e0_test.index]

after = pd.DataFrame([
    {"Candidate": "Pooled, no indicators", "Reference": "EPL-only", "Matches": "EPL",
     **compare(e0_test, "Pooled, no indicators", "EPL-only")},
    {"Candidate": "Pooled, no indicators", "Reference": "Per league", "Matches": "All five",
     **compare(test, "Pooled, no indicators", "Per league")},
    {"Candidate": "Hybrid", "Reference": "Per league", "Matches": "All five",
     **compare(test, "Hybrid", "Per league")},
])
after

,Candidate,Reference,Matches,matches,mean,ciLow,ciHigh,pBetter,seasonsBetter,better,nonInferior
0,"Pooled, no indicators",EPL-only,EPL,1520,-0.0014,-0.0050,0.0020,0.7882,2,False,True
1,"Pooled, no indicators",Per league,All five,7082,-0.0027,-0.0049,-0.0006,0.9936,4,True,True
2,Hybrid,Per league,All five,7082,-0.0029,-0.0044,-0.0014,0.9998,4,True,True



## Conclusion

**Gate 1 fails, so the pooled model doesn't ship as planned.** On the 1,520
Premier League test matches, pooled vs today's EPL-only model is
**-0.0001 [-0.0028, +0.0025]**: P(better) 0.53, better in 2 of 4 seasons. The
pooled model is no worse on average, but the upper bound misses the +0.002
non-inferiority margin.

**Gate 2 passes, replicating notebook 18 on the pipeline's own data.**
Pooled vs per league over all 7,082 top-5 test matches is
**-0.0029 [-0.0046, -0.0013]**, better in all 4 seasons (nb 18:
-0.0028 [-0.0044, -0.0012]). The gain is in the other leagues, and the
Premier League is unchanged:

| League | Per league | Pooled | Pooled - per league [95% CI] | Bookmaker |
|---|---|---|---|---|
| Premier League | 0.9799 | 0.9798 | -0.0001 [-0.0028, +0.0025] | 0.9605 |
| La Liga | 0.9911 | 0.9847 | -0.0064 [-0.0108, -0.0020] | 0.9595 |
| Bundesliga | 0.9991 | 0.9935 | -0.0056 [-0.0105, -0.0008] | 0.9697 |
| Serie A | 0.9948 | 0.9928 | -0.0020 [-0.0049, +0.0009] | 0.9670 |
| Ligue 1 | 0.9989 | 0.9982 | -0.0007 [-0.0034, +0.0022] | 0.9784 |
| **All five** | **0.9923** | **0.9894** | **-0.0029 [-0.0046, -0.0013]** | 0.9666 |

**Why gate 1 failed: it can't be passed by a model that is merely as good.**
The per-match difference has SD 0.0515, so on 1,520 matches the CI
half-width is about ±0.0026, wider than the 0.002 margin. To pass, the pooled
model would have to be **better** on the EPL by more than 0.0006, or the test
would need about 2,550 EPL matches. The plan sized the margin from the
pooled-vs-per-league interval over all 7,082 matches, not the EPL's 1,520.

**Options (the owner's decision; nothing here changes `ENABLED`):**

1. **Hybrid: today's EPL-only model for the Premier League, pooled for the
   other four.** EPL predictions are exactly today's, so gate 1 holds by
   construction. Over all five leagues it keeps the whole gain:
   -0.0029 [-0.0044, -0.0014], better in 4/4 seasons. The cost is two models
   to train each run instead of one.
2. **Accept the pooled model on the evidence.** The mean EPL change is -0.0001,
   and P1-04 showed the EPL's features are bit-identical either way. This
   means overruling the gate (or widening the margin for gates on a single
   league's ~1,500 matches), which only the owner can do.
3. **Pooled without the league indicators.** It passes gate 1 by a hair
   (+0.0020 upper bound, -0.0014 mean) and still passes gate 2
   (-0.0027 [-0.0049, -0.0006]). But it was picked **after** gate 1 failed,
   the indicators-vs-none difference is noise (-0.0002 [-0.0014, +0.0010]),
   and a pass this close is a coin flip on resampling. Not recommended.

**Other findings:**
- **Flat-stake ROI** at average opening odds rises from -2.9% to -2.0% pooled.
  La Liga goes from -1.4% to +1.6%; the Premier League stays at +0.4%.
- **Indicator coefficients** are small and plausible: Serie A has more draws
  and fewer home wins; La Liga has fewer away wins.
- **xG coverage** is 100% in every league-season, all FotMob except one
  Understat match (SP1 23-24, Mallorca v Girona).


**Decision (owner, 2026-11-03): option 1, the hybrid.** Each competition's
`model` in `competitions.py` is `"own"` for E0 and `"pooled"` for the rest;
`model.train_models` fits both. The pipeline's own backtest gives exactly
this notebook's hybrid: the E0 section is identical to the EPL-only
backtest, and the other leagues have their pooled figures.
